In [5]:
import torch
import torchvision
import math
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F

device=torch.device("cuda:0")

cfg=dict(
    T=1000, beta_start=1e-4,beta_end=0.02,
    img_size=32, channels=3, 
    base_channels=128, channel_mults=(1,2,2,2), num_res_blocks=2, attn_resolution=16, dropout=0.1,
    batch_size=128, lr=2e-4, ema_decay=0.9999,
    sample_every=5000, ckpt_every=5000,
)
torch.manual_seed(0)
print(cfg)
print(torch.cuda.get_device_name())
print(torch.__version__)


{'T': 1000, 'beta_start': 0.0001, 'beta_end': 0.02, 'img_size': 32, 'channels': 3, 'base_channels': 128, 'channel_mults': (1, 2, 2, 2), 'num_res_blocks': 2, 'attn_resolution': 16, 'dropout': 0.1, 'batch_size': 128, 'lr': 0.0002, 'ema_decay': 0.9999, 'sample_every': 5000, 'ckpt_every': 5000}
NVIDIA GeForce RTX 5090
2.11.0+cu128


In [8]:
from torch.utils.data import DataLoader


transforms=torchvision.transforms.Compose(
    [torchvision.transforms.RandomHorizontalFlip(), torchvision.transforms.ToTensor(), torchvision.transforms.Normalize(0.5,0.5)])

dataset=torchvision.datasets.CIFAR10(root="./data",train=True, download=True, transform=transforms)

train_dataloader=DataLoader(dataset, cfg["batch_size"], shuffle=True,num_workers=8, pin_memory=True, drop_last=True)

x0_demo, _=next(iter(train_dataloader))
    
    

/home/c504-5090-x2/miniconda3/lib/python3.13/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [9]:
T = cfg["T"]
beta = torch.linspace(cfg["beta_start"], cfg["beta_end"], T, dtype=torch.float64)
alpha = 1.0 - beta                       
alpha_bar = torch.cumprod(alpha, dim=0)  

sqrt_ab   = alpha_bar.sqrt().float().to(device)        
sqrt_1mab = (1 - alpha_bar).sqrt().float().to(device)  

print(f"ᾱ_1={alpha_bar[0]:.6f}  ᾱ_500={alpha_bar[499]:.4f}  ᾱ_1000={alpha_bar[-1]:.2e}")


ᾱ_1=0.999900  ᾱ_500=0.0786  ᾱ_1000=4.04e-05


In [10]:
def q_sample(x0, t, eps):
    """x0: (B,3,32,32) in [-1,1] | t: (B,) long on device | eps: randn like x0"""
    return (sqrt_ab[t].view(-1, 1, 1, 1) * x0 +
            sqrt_1mab[t].view(-1, 1, 1, 1) * eps)

In [13]:
def timestep_embed(t, dim):
    t=t.float()
    freqs=torch.exp(-math.log(10000)*torch.arange(dim//2, device=t.device)/(dim//2))
    outer_product=t[:,None]*freqs[None,:]
    output=torch.cat([torch.sin(outer_product), torch.cos(outer_product)], dim=-1)
    return output
    
class TimeEmbedding(nn.Module):
    def __init__(self, base_channels, time_dim):
        super().__init__()
        self.base_channels=base_channels
        self.time_dim=time_dim
        
        self.linear1=nn.Linear(base_channels, time_dim)
        self.silu=nn.SiLU(inplace=False)
        self.linear2=nn.Linear(time_dim,time_dim)
    def forward(self, t):
        t=timestep_embed(t, self.base_channels)
        t=self.linear1(t)
        t=self.silu(t)
        t=self.linear2(t)
        return t    

In [15]:
class ResBlock(nn.Module):
    def __init__(self, time_dim, in_ch, out_ch,cfg):
        super().__init__()
        assert in_ch % 32 == 0 and out_ch % 32 == 0
        self.time_dim=time_dim
        self.out_ch=out_ch
        self.in_ch=in_ch
        
        self.norm1=nn.GroupNorm(32, in_ch)
        self.silu=nn.SiLU(inplace=False)
        self.conv1=nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1)
        self.linear_proj=nn.Linear(time_dim, 2*out_ch)
        self.norm2=nn.GroupNorm(32, out_ch)
        self.dropout=nn.Dropout(cfg["dropout"])
        self.conv2=nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1)
        if in_ch==out_ch:
            self.res=nn.Identity()
        else:
            self.res=nn.Conv2d(in_ch, out_ch, kernel_size=1)

    def forward(self, x, t_emb):
        h=self.norm1(x)
        h=self.silu(h)
        h=self.conv1(h)

        proj=self.silu(t_emb)
        proj=self.linear_proj(proj)
        scale, shift=proj.chunk(2, dim=1)

        scale=scale.view(-1, self.out_ch, 1 ,1)
        shift=shift.view(-1,self.out_ch, 1, 1)

        h=self.norm2(h)*(1+scale)+shift
        h=self.silu(h)
        h=self.dropout(h)
        h=self.conv2(h)

        return h+self.res(x)
    
        
        
        
        
    

In [19]:
class AttentionBlock(nn.Module):
    def __init__(self, channels, num_heads=None):
        super().__init__()
        if num_heads is None:
            assert channels % 64 == 0
            num_heads = channels // 64          # derive FIRST
        assert channels % num_heads == 0
        self.num_heads = num_heads              # store AFTER deriving
        self.head_dim = channels // num_heads
        self.norm = nn.GroupNorm(32, channels)
        self.proj = nn.Conv2d(channels, channels * 3, 1)
        self.out = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        h = self.norm(x)
        qkv = self.proj(h)
        q, k, v = qkv.chunk(3, dim=1)
        B, C, H, W = q.shape
        # split channels into heads (legal reshape: splits adjacent axis)...
        q = q.reshape(B, self.num_heads, self.head_dim, H * W).transpose(2, 3)
        k = k.reshape(B, self.num_heads, self.head_dim, H * W).transpose(2, 3)
        v = v.reshape(B, self.num_heads, self.head_dim, H * W).transpose(2, 3)
        # ...then transpose to (B, heads, seq, head_dim) for SDPA
        h = F.scaled_dot_product_attention(q, k, v)
        h = h.transpose(2, 3).reshape(B, C, H, W)   # exact inverse choreography
        h = self.out(h)
        return x + h
        

In [ ]:
class UNet(nn.Module):
    def __init__(self, ):
        super().__init__()
        self.stem=nn.Conv2d(3,128, 3,1)
        self.resblock1=ResBlock(
        